In [1]:
# Installing pandasql
import subprocess
subprocess.run(["pip", "install", "pandasql"])

CompletedProcess(args=['pip', 'install', 'pandasql'], returncode=0)

In [2]:
import pandas as pd
import sqlite3
import os

In [4]:
df = pd.read_csv(r"C:\Users\Soumili Nag\Downloads\projects\NovaPay-AML-Analysis\data\PS_20174392719_1491204439457_log.csv.zip")

In [8]:
# Renaming the columns to be cleaner and professional
df.columns = [
    'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
    'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest',
    'isFraud', 'isFlaggedFraud'
]

In [5]:
print("Shape:", df.shape)

Shape: (6362620, 11)


In [7]:
print("\nColumn Names:", df.columns.tolist())


Column Names: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [11]:
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [13]:
# Column reference for understanding the data
column_guide = {
    'step':           'Hour of transaction (1 step = 1 hour, max 744 = 30 days)',
    'type':           'Transaction type: CASH-IN, CASH-OUT, DEBIT, PAYMENT, TRANSFER',
    'amount':         'Amount of the transaction in local currency',
    'nameOrig':       'Customer who initiated the transaction',
    'oldbalanceOrg':  'Sender balance BEFORE transaction',
    'newbalanceOrig': 'Sender balance AFTER transaction',
    'nameDest':       'Recipient of the transaction',
    'oldbalanceDest': 'Recipient balance BEFORE transaction',
    'newbalanceDest': 'Recipient balance AFTER transaction',
    'isFraud':        '1 = This is a fraudulent transaction (ground truth)',
    'isFlaggedFraud': '1 = System flagged this as fraud (often wrong = false positive)'
}

In [14]:
for col, desc in column_guide.items():
    print(f"  {col:<20} → {desc}")

  step                 → Hour of transaction (1 step = 1 hour, max 744 = 30 days)
  type                 → Transaction type: CASH-IN, CASH-OUT, DEBIT, PAYMENT, TRANSFER
  amount               → Amount of the transaction in local currency
  nameOrig             → Customer who initiated the transaction
  oldbalanceOrg        → Sender balance BEFORE transaction
  newbalanceOrig       → Sender balance AFTER transaction
  nameDest             → Recipient of the transaction
  oldbalanceDest       → Recipient balance BEFORE transaction
  newbalanceDest       → Recipient balance AFTER transaction
  isFraud              → 1 = This is a fraudulent transaction (ground truth)
  isFlaggedFraud       → 1 = System flagged this as fraud (often wrong = false positive)


In [18]:
# Create a local SQL database from your CSV
conn = sqlite3.connect(r'C:\Users\Soumili Nag\Downloads\projects\NovaPay-AML-Analysis\data\novapay_aml.db')

In [17]:
# Load the dataframe into the database as a table called 'transactions'
df.to_sql('transactions', conn, if_exists='replace', index = False)

6362620

In [19]:
print("✅ Database created successfully!")
print(f"   Table 'transactions' loaded with {len(df):,} rows")

✅ Database created successfully!
   Table 'transactions' loaded with 6,362,620 rows


In [20]:
# Save connection for use in later cells
conn.close()

In [25]:
# Helper function — write SQL and see results as a clean table
def run_sql(query):
    conn = sqlite3.connect(r'C:\Users\Soumili Nag\Downloads\projects\NovaPay-AML-Analysis\data\novapay_aml.db')
    result = pd.read_sql_query(query, conn)
    conn.close()
    return result

In [24]:
# First SQL query — count all transactions by type
result = run_sql("""
    SELECT 
        type,
        COUNT(*) AS total_transactions,
        ROUND(SUM(amount), 2) AS total_amount,
        SUM(isFraud) AS fraud_count
    FROM transactions
    GROUP BY type
    ORDER BY total_transactions DESC
""")

result

,type,total_transactions,total_amount,fraud_count
0,CASH_OUT,2237500,3.944130e+11,4116
1,PAYMENT,2151495,2.809337e+10,0
2,CASH_IN,1399284,2.363674e+11,0
3,TRANSFER,532909,4.852920e+11,4097
4,DEBIT,41432,2.271992e+08,0
